# Libra

> Brady Spears, Los Alamos National Laboratory

## Serializing & Deserializing from a Database Connection

---
### About this Notebook
This notebook provides insight into the simple instantiation of `Libra's` _Schema_ object, and gives examples of how to 'load' and 'dump' abstract `SQLAlchemy` _Table_ instances (referred to interchangably as _Models_) to and from a relational database. Some `SQLAlchemy` boilerplate code is shown to exemplify how a user can interact with `SQLAlchemy` declarative objects, performing CRUD (create-read-update-delete) operations in the OOP environment that are then reflected in the SQL envrionment.

---
### Defining Schema definition tables
A "schema definition" is a serialized instance of a relational database schema, and in this case, it is contained in a set of relational database tables. These tables describe the schema, models belonging to the schema, columns, and various constraints. The National Nuclear Security Administration's "Knowledge Base" relational database schema is the schema we'll be interacting with. First, we need to create the database we'll be working with and push the schema definition up there. For this, we'll be loading the NNSA KB Core schema from a YAML file (another `Libra` built-in definition format) and exporting it to our SQLite3 database:

In [3]:
import os

import sqlalchemy
from sqlalchemy.orm import sessionmaker

from libra import Schema
from libra.util import DatabaseSettings

db_filepath = 'data/examples.db'
yaml_filepath = 'data/kbcore.yaml'

# Clean up any existing database files
if os.path.exists(db_filepath):
    os.remove(db_filepath)

# Load NNSA KB Core in from YAML file
kbcore = Schema('NNSA KB Core').load(yaml_filepath)

# Get a SQLAlchemy Engine
engine = sqlalchemy.create_engine(f'sqlite:///{db_filepath}')

# Initialialize a DatabaseSettings object
db_settings = DatabaseSettings(
    engine = engine,
    author = 'NNSA', # Used for administration-level tracking in schema def tables
    create_tables = True # Create the tables if we need to
)

# Dump KB Core into the database
kbcore.dump(db_settings)

We've populated a series of relational database tables that act to describe `SQLAlchemy` models. We can then subsequently load the NNSA KB Core schema back using the information contained in those tables. To do so, just pass the same 'db_settings' object to the 'load()' method now. `Libra` will establish a connection to the database and construct a schema definition dictionary from the information contained therein. 

In [5]:
kbcore_new = Schema('NNSA KB Core').load(db_settings)

print(f'Schema \'{kbcore.name}\' contains :')
print(f'  Models : {list(kbcore._registry.models.keys())}')
print(f'  Columns : {list(kbcore._registry.columns.keys())}')

Schema 'NNSA KB Core' contains :
  Models : ['affiliation', 'amplitude', 'arrival', 'assoc', 'event', 'gregion', 'instrument', 'lastid', 'netmag', 'network', 'origerr', 'origin', 'remark', 'sensor', 'site', 'sitechan', 'sregion', 'stamag', 'wfdisc', 'wftag']
  Columns : ['algorithm', 'amp', 'ampid', 'amptime', 'amptype', 'arid', 'auth', 'azdef', 'azimuth', 'azres', 'band', 'belief', 'calib', 'calper', 'calratio', 'chan', 'chanid', 'clip', 'commid', 'conf', 'ctype', 'datatype', 'deast', 'delaz', 'delslo', 'delta', 'deltaf', 'deltim', 'depdp', 'depth', 'descrip', 'dfile', 'digital', 'dir', 'dnorth', 'dtype', 'duration', 'edepth', 'elev', 'ema', 'emares', 'endtime', 'esaz', 'etype', 'evid', 'evname', 'fm', 'foff', 'grn', 'grname', 'hang', 'inarrival', 'inid', 'insname', 'instant', 'instype', 'iphase', 'jdate', 'keyname', 'keyvalue', 'lat', 'lddate', 'lineno', 'logat', 'lon', 'magdef', 'magid', 'magnitude', 'magres', 'magtype', 'mb', 'mbid', 'ml', 'mlid', 'mmodel', 'ms', 'msid', 'nass', 'n

Through the _Schema_ object, concrete (not abstract) `SQLAlchemy` _Table_ objects can be created by simply creating a new class tha inherits from the Model and has a '__tablename__' attribute attached to it. The '__tablename__' attribute is the name of the table in the SQL environment and what `SQLAlchemy` uses to reference the table when rendering Data Definition Language (DDL). This DDL is emitted anytime a transaction with the database backend occurs, whether that is creating tables, dropping tables, inserting data, or updating data.

In [6]:
class Netmag(kbcore.netmag):
    __tablename__ = 'my_netmag_table'

# All methods and components of the Event object
print(Netmag.__table__.__dict__)

{'dispatch': <sqlalchemy.event.base.DDLEventsDispatch object at 0x7f1b4fb09fd0>, 'name': 'my_netmag_table', '_columns': <sqlalchemy.sql.base.DedupeColumnCollection object at 0x7f1b4fc5f840>, 'primary_key': PrimaryKeyConstraint(Column('magid', Numeric(precision=9), table=<my_netmag_table>, primary_key=True, nullable=False, default=ScalarElementColumnDefault(-1))), 'foreign_keys': set(), 'fullname': 'my_netmag_table', 'metadata': MetaData(), 'schema': None, '_sentinel_column': None, 'indexes': set(), 'constraints': {UniqueConstraint(Column('auth', String(length=20), table=<my_netmag_table>, default=ScalarElementColumnDefault('-')), Column('magtype', String(length=6), table=<my_netmag_table>, default=ScalarElementColumnDefault('-')), Column('orid', Numeric(precision=9), table=<my_netmag_table>, default=ScalarElementColumnDefault(-1))), PrimaryKeyConstraint(Column('magid', Numeric(precision=9), table=<my_netmag_table>, primary_key=True, nullable=False, default=ScalarElementColumnDefault(-1

Now that we have a concrete table, we can use standard `SQLAlchemy` declarative syntax to interact with it. Documentation on how to interact with declarative objects in `SQLAlchemy` is available at https://www.sqlalchemy.org/

In [7]:
from datetime import datetime

from sqlalchemy.orm import sessionmaker

LocalSession = sessionmaker(bind = engine)

with LocalSession() as session:

    # Create all tables associated with kbcore_subset Schema object
    kbcore.base.metadata.create_all(engine)

    # Create a list of objects to add to the database
    entries = [
        Netmag(1, 'NET', 14, 5, 'mb', 20, 4.5, None, 'USGS', None, None),
        Netmag(2, 'NET', 16, 7, 'mb', 20, 4.2, None, 'USGS', None, None),
        Netmag(3, 'TX', 17, 8, 'ms', 52, 4.1, None, 'TXNET', None, None),
        Netmag(magid = 4, net = 'TX', orid = 18, evid = 9, magtype = 'ml', nsta = 31, magnitude = 5.0, auth = 'TXNET')
    ]

    [session.add(entry) for entry in entries]
    session.commit() # Commit the additions to the database

    # Now, let's query all of that back
    query_results = session.query(Netmag).all()

    # Also, show an example of filtering
    filtered_query_results = session.query(Netmag).filter(Netmag.magtype == 'mb').all()

print('All query results:')
for row in query_results:
    print(f'  {row}')

print('Filtered query results:')
for row in filtered_query_results:
    print(f'  {row}')

All query results:
  Netmag(magid=1.0000000000, net='NET', orid=14.0000000000, evid=5.0000000000, magtype='mb', nsta=20.0000000000, magnitude=4.5, uncertainty=-1.0, auth='USGS', commid=-1.0000000000, lddate=2026-03-03 16:12:57.070961)
  Netmag(magid=2.0000000000, net='NET', orid=16.0000000000, evid=7.0000000000, magtype='mb', nsta=20.0000000000, magnitude=4.2, uncertainty=-1.0, auth='USGS', commid=-1.0000000000, lddate=2026-03-03 16:12:57.071008)
  Netmag(magid=3.0000000000, net='TX', orid=17.0000000000, evid=8.0000000000, magtype='ms', nsta=52.0000000000, magnitude=4.1, uncertainty=-1.0, auth='TXNET', commid=-1.0000000000, lddate=2026-03-03 16:12:57.071040)
  Netmag(magid=4.0000000000, net='TX', orid=18.0000000000, evid=9.0000000000, magtype='ml', nsta=31.0000000000, magnitude=5.0, uncertainty=-1.0, auth='TXNET', commid=-1.0000000000, lddate=2026-03-03 16:12:57.071119)
Filtered query results:
  Netmag(magid=1.0000000000, net='NET', orid=14.0000000000, evid=5.0000000000, magtype='mb', 

Notice how above, in defining the _Netmag_ object, `Libra` supports both positional and keyword initialization patterns. For values that are _None_ or simply not included, `Libra's` custom metaclass defaults that column's value to the pre-defined 'default' parameter. 